Random Forest Implementation

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    roc_auc_score, average_precision_score, log_loss, 
    classification_report, confusion_matrix
)
from CartClassifier import CARTClassifier

#Random Forest Implementation:


class RandomForestFromScratch:
    def __init__(self, n_estimators=10, max_depth=10, min_samples_split=2, min_samples_leaf=1):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.trees = []

    def _bootstrap_sample(self, X, y):
        n_samples = X.shape[0]
        # Sampling with replacement
        indices = np.random.choice(n_samples, n_samples, replace=True)

        return X.iloc[indices].values, y[indices]

    def fit(self, X, y):
        self.trees = []
        for i in range(self.n_estimators):
            X_sample, y_sample = self._bootstrap_sample(X, y)
            
            # Initialize CART
            tree = CARTClassifier(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                min_samples_leaf=self.min_samples_leaf
            )
            
            # Train the individual tree
            tree.fit(X_sample, y_sample)
            self.trees.append(tree)

    def predict_proba(self, X):
        #Calculates probabilities for AUROC, AUPRC, and Log-Loss
        # Convert X to values if it's a DataFrame
        X_vals = X.values if isinstance(X, pd.DataFrame) else X
        
        # Collect predictions (0 or 1) from every tree
        # Shape: (n_estimators, n_samples)
        all_tree_preds = np.array([tree.predict(X_vals) for tree in self.trees])
        
        # The probability of Class 1 is the average of tree votes
        prob_pos = np.mean(all_tree_preds, axis=0)
        
        # Return format: [prob_of_0, prob_of_1]
        return np.vstack([1 - prob_pos, prob_pos]).T

    def predict(self, X):
        #Hard label prediction via majority vote
        probas = self.predict_proba(X)
        return np.argmax(probas, axis=1)

   # Evaluates Accuracy, Precision, Recall, F1, Log-Loss, AUROC, and AUPRC.
def run_performance_assessment(model, X, y, set_name="Test", show_report=False):
   
    y_pred = model.predict(X)
    y_prob = model.predict_proba(X)[:, 1]
    
    # Calculate all metrics
    metrics = {
        "Accuracy": accuracy_score(y, y_pred),
        "Precision": precision_score(y, y_pred),
        "Recall": recall_score(y, y_pred),
        "F1-Score": f1_score(y, y_pred),
        "Log-Loss": log_loss(y, y_prob),
        "ROC-AUC": roc_auc_score(y, y_prob),
        "AUPRC": average_precision_score(y, y_prob)
    }
    
    # Display header
    print(f"-- {set_name} --")
    
    # Print metrics
    for name, value in metrics.items():
        print(f"{name:10}: {value:.4f}")
    
    # Print classification report
    if show_report:
        print("\nClassification Report:")
        print(classification_report(y, y_pred))
    
    print("-" * 30 + "\n")
    return metrics


In [ ]:
# 1. Load data
df = pd.read_csv("../data/features/processed_data_with_position_specific_features.csv")

# 2. Preprocess (Drop non-feature strings)
DROP_COLS = ["peptide", "hla_sequence", "index", "HLA"]
TARGET_COL = "Label"
df = df.drop(columns=[c for c in DROP_COLS if c in df.columns])

# 3. Handle categorical columns (Encoding)
cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
cat_cols = [c for c in cat_cols if c != TARGET_COL]
df = pd.get_dummies(df, columns=cat_cols)

# 4. Define Features (X) and Target (y)
y = df[TARGET_COL].values
X = df.drop(columns=[TARGET_COL])

# 5. Load pre-defined split indices
train_idx = np.load("../data/splits/train_idx.npy")
val_idx   = np.load("../data/splits/val_idx.npy")
test_idx  = np.load("../data/splits/test_idx.npy")

X_train, y_train = X.iloc[train_idx], y[train_idx]
X_val, y_val     = X.iloc[val_idx], y[val_idx]
X_test, y_test   = X.iloc[test_idx], y[test_idx]

# 6. Initialize and Fit Random Forest
# Increased n_estimators to 50 for more stable probabilities
rf_model = RandomForestFromScratch(n_estimators=50, max_depth=10)
rf_model.fit(X_train, y_train)

# 7. Final Performance Assessment (Train, Validation, and Test)

train_results = run_performance_assessment(rf_model, X_train, y_train, "Training Set")
val_results   = run_performance_assessment(rf_model, X_val, y_val, "Validation Set")
test_results  = run_performance_assessment(rf_model, X_test, y_test, "Test Set", show_report=True)



===== TRAINING SET PERFORMANCE =====
Accuracy    : 0.8898
Precision   : 0.8522
Recall      : 0.9130
F1-Score    : 0.8816
Log-Loss    : 0.2610
AUROC       : 0.9641
AUPRC       : 0.9583

===== VALIDATION SET PERFORMANCE =====
Accuracy    : 0.7787
Precision   : 0.7390
Recall      : 0.8054
F1-Score    : 0.7708
Log-Loss    : 0.7030
AUROC       : 0.8566
AUPRC       : 0.8177

===== TEST SET PERFORMANCE =====
Accuracy    : 0.7961
Precision   : 0.7462
Recall      : 0.8350
F1-Score    : 0.7881
Log-Loss    : 0.6080
AUROC       : 0.8721
AUPRC       : 0.8216


In [ ]:
def evaluate_independent_set(file_path, display_name):
    df_ind = pd.read_csv(file_path)
    
    # Preprocess to match training structure
    TARGET_COL = "Label"
    X_ind = df_ind.drop(columns=[TARGET_COL] if TARGET_COL in df_ind.columns else [])
    X_ind = pd.get_dummies(X_ind)
    
    X_ind = X_ind.reindex(columns=X.columns, fill_value=0)
    y_ind = df_ind[TARGET_COL].values
    
    # Run assessment
    return run_performance_assessment(rf_model, X_ind, y_ind, display_name, show_report=True)

In [ ]:
# Evaluate Dengue
evaluate_independent_set("../data/independent_tests/dengue_with_position_specific_features.csv", "Dengue")

# Evaluate SARS-CoV-2
evaluate_independent_set("../data/independent_tests/sars_cov_2_with_position_specific_features.csv", "SARS-CoV-2")

# Evaluate Zika
evaluate_independent_set("../data/independent_tests/zika_with_position_specific_features.csv", "Zika")

In [ ]:
# 1. Define the grid of hyperparameters
depth_options = [5, 10, 15]
split_options = [2, 5, 10]
leaf_options  = [1, 2, 5]

best_auc = 0
best_params = {}

# 2. Loop through every combination
for depth in depth_options:
    for split in split_options:
        for leaf in leaf_options:
            # Train model
            temp_model = CARTClassifier(max_depth=depth, min_samples_split=split, min_samples_leaf=leaf)
            temp_model.fit(X_train, y_train)
            
            # Evaluate on Validation Set (the gold standard for tuning!)
            val_preds = temp_model.predict_proba(X_val)[:, 1]
            current_auc = roc_auc_score(y_val, val_preds)
            
            # Keep track of the best one
            if current_auc > best_auc:
                best_auc = current_auc
                best_params = {'max_depth': depth, 'min_samples_split': split, 'min_samples_leaf': leaf}

print(f"Best Validation AUC: {best_auc:.4f}")
print(f"Best Parameters: {best_params}")